# Fig. S1A: UMAP (cell type + pseudotime)

Adult mouse SVZ neurogenesis. Left: lineage cells coloured by cell type
(`paper_palette_celltypes`); right: lineage cells coloured by diffusion
pseudotime (magma). Non-lineage cells are not shown.

- Kernel: `apa-python`.
- Input: `wildtypes_raw_lognorm_scaled_batch_corrected_ptime_celltypes_htos.h5ad`.
- Output: `$M26_RESULTS_ROOT/Fig1/fig1a_umap/` (svg + png).


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# shared paper style (palette + fonts + save/display helper)
STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

PANEL = "fig1a_umap"
PANEL_W_PX, PANEL_H_PX = S.HALF_W_PX, S.mm_to_px(50)  # 85 x 50 mm (half-grid)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)

SCRNA_PATH = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/"
    "wildtypes_raw_lognorm_scaled_batch_corrected_ptime_celltypes_htos.h5ad"
)
print("lineage order:", S.MOUSE_LINEAGE_ORDER)
print("palette:", {k: S.paper_palette_celltypes[k] for k in S.MOUSE_LINEAGE_ORDER})

In [ ]:
adata_sc = sc.read_h5ad(SCRNA_PATH)
print(f"scRNA-seq: {adata_sc.shape[0]:,} cells x {adata_sc.shape[1]:,} genes")

lineage_mask = adata_sc.obs["celltype"].isin(S.MOUSE_LINEAGE_ORDER)
n_lineage = int(lineage_mask.sum())
n_mice = int(adata_sc.obs.loc[lineage_mask, "batch"].nunique())
print(f"lineage cells: {n_lineage:,} | mice: {n_mice}")

In [ ]:
import numpy as np
from matplotlib.lines import Line2D

SEED = 0  # fixed seed -> reproducible random plotting order

# Lineage cells only: the non-lineage ("grey") cells are dropped, as the paper
# only discusses the neurogenic lineage. Coordinates + labels for the kept cells:
fg = lineage_mask.values
xy = adata_sc.obsm["X_umap"][fg]
ct_lin = adata_sc.obs.loc[fg, "celltype"].astype(str).values
pt_lin = adata_sc.obs.loc[fg, "dpt_pseudotime"].values.astype(float)

# Per-mouse cell counts for the pie inset; "batch" column = bio-replicate
# (5-8 mice pooled per batch; no clean per-individual-mouse obs column).
mouse_lin = adata_sc.obs.loc[fg, "batch"].astype(str).values
mice = sorted(np.unique(mouse_lin), key=int)
mouse_counts = np.array([(mouse_lin == m).sum() for m in mice])
# Set2: muted/desaturated palette - visually distinct from the vivid cell-type colours
MOUSE_COLORS = plt.cm.Set2(np.linspace(0, 1, len(mice)))


def _emptiest_corner(pts):
    """Inset bbox [x0, y0, w, h] in the emptiest free corner of the point cloud.
    Lower-left holds the cell-type legend and upper-left the n-annotation, so the pie
    is placed in whichever of lower-right / upper-right is sparser."""
    xm, ym = np.median(pts[:, 0]), np.median(pts[:, 1])
    quad = {
        "lower right": int(((pts[:, 0] >= xm) & (pts[:, 1] < ym)).sum()),
        "upper right": int(((pts[:, 0] >= xm) & (pts[:, 1] >= ym)).sum()),
    }
    loc = {"lower right": [0.62, 0.02], "upper right": [0.62, 0.60]}
    return loc[min(quad, key=quad.get)] + [0.24, 0.24]


def render_fig1a(point_size, panel, seed=SEED):
    """Render the two-panel UMAP (cell type + pseudotime), lineage cells only,
    plotted in random order, with a cells-per-mouse pie inset."""
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, constrained_layout=True)

    # panel 1 - cell type (random plotting order)
    rgba_ct = S.palette_rgba(ct_lin, S.paper_palette_celltypes, alpha=0.85)
    S.scatter_random_order(
        axes[0], xy[:, 0], xy[:, 1], rgba=rgba_ct, s=point_size, seed=seed
    )
    handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.paper_palette_celltypes[c],
            markersize=3,
            label=c,
        )
        for c in S.MOUSE_LINEAGE_ORDER
    ]
    axes[0].legend(
        handles=handles,
        fontsize=S.FONT_LEGEND,
        frameon=False,
        loc="lower left",
        handletextpad=0.2,
        borderpad=0.2,
        labelspacing=0.2,
    )
    axes[0].set_title("Cell type", fontsize=S.FONT_LABEL, pad=2)

    # pie inset - proportion of lineage cells contributed by each mouse
    axpie = axes[0].inset_axes(_emptiest_corner(xy))
    axpie.pie(
        mouse_counts,
        colors=MOUSE_COLORS,
        startangle=90,
        counterclock=False,
        wedgeprops=dict(edgecolor="white", linewidth=0.3),
    )
    axpie.set_title(
        f"cells per mouse\n(n={len(mice)})", fontsize=S.FONT_ANNOT - 1.5, pad=1
    )

    # panel 2 - pseudotime (random plotting order)
    sm = S.scatter_random_order(
        axes[1],
        xy[:, 0],
        xy[:, 1],
        c=pt_lin,
        cmap=S.PTIME_CMAP,
        s=point_size,
        seed=seed,
    )
    cbar = fig.colorbar(sm, ax=axes[1], shrink=1.0, aspect=20, pad=0.02)
    cbar.set_label("Pseudotime", fontsize=S.FONT_TICK)
    cbar.ax.tick_params(labelsize=S.FONT_TICK - 1)
    axes[1].set_title("Pseudotime", fontsize=S.FONT_LABEL, pad=2)

    for ax in axes:
        ax.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL)
        ax.set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
        ax.set_xticks([])
        ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)

    axes[0].text(
        0.02,
        0.98,
        f"n = {n_lineage:,} lineage cells\n{n_mice} mice",
        transform=axes[0].transAxes,
        fontsize=S.FONT_ANNOT,
        va="top",
        ha="left",
    )
    S.save_and_show(fig, panel)


render_fig1a(2.0, PANEL)

### Variation: reduced point size

Same UMAP as above but with smaller points, still plotted in random cell order (fixed
seed). Saved as `fig1a_umap__smallpts`.


In [ ]:
# Reduced-point-size variation (random order); saved alongside the default-size panel.
render_fig1a(0.8, PANEL + "__smallpts")